# Assignment 08 — Classification Models Comparison

**Student:** Sayem Islam 
**Dataset:** `disease_risk_dataset.csv` (400 patients; 1 = high risk). Run top to bottom with the CSV beside this notebook or in `upload/`.

**Goal:** compare Logistic Regression, linear/RBF SVM, Gaussian Naive Bayes, and Decision Tree on the same untouched stratified test set. 5-fold training-only CV tunes hyperparameters. This is coursework on a sample dataset, not a validated clinical screening system.

## Executed check — quick results

400 patients; 0 duplicate rows; `bmi`, `cholesterol`, `blood_sugar` each have 8 missing entries. Stratified training/test sizes: 320/80. On the held-out set Logistic Regression accuracy = **0.75** and high-risk recall = **0.75**; Decision Tree accuracy = **0.5375** and high-risk recall = **0.3611**. Run all cells for the complete CV and five-model comparison, graphs, and the automatically generated 270-word discussion.

## Phase 0 — Load and inspect duplicates (before split)
Duplicates are checked before modeling; removing exact repeated rows before the train/test split avoids identical patients appearing in both sets. A repeated patient record might be legitimate in a longitudinal dataset, but this dataset has one row per patient.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             roc_auc_score, roc_curve)

sns.set_theme(style="whitegrid")
DATA_PATH = next((path for path in [Path('disease_risk_dataset.csv'), Path('upload/disease_risk_dataset.csv'),
                        Path('../disease_risk_dataset.csv')] if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError('Place disease_risk_dataset.csv beside this notebook or in upload/.')
df = pd.read_csv(DATA_PATH)
print('Source:', DATA_PATH, '| Original shape:', df.shape)
print('Exact duplicate rows:', df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print('Shape after deduplication:', df.shape)
display(df.head())

## Phase 1 — Understanding + EDA 
We inspect types, summary statistics, missingness, class balance and two feature/target relationships. The target is used here for descriptive plots only; no preprocessing parameter is fitted to the full dataset. The CSV has numeric values and no `?` placeholders, but the replacement below makes the check explicit.

In [ ]:
df = df.replace('?', np.nan)
print('Shape:', df.shape)
print('Types:'); display(df.dtypes.to_frame('dtype'))
print('Description:'); display(df.describe(include='all').T)
print('Missing per column:'); display(df.isna().sum().to_frame('missing_count'))
print('Class counts:'); display(df.disease_risk.value_counts().sort_index().rename(index={0:'Low risk',1:'High risk'}))
assert df.disease_risk.notna().all() and set(df.disease_risk.unique()) == {0, 1}


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=df, x='disease_risk', hue='disease_risk', palette='Set2', legend=False, ax=ax)
ax.set(title='Patient count by disease risk', xlabel='Disease risk (0 = low, 1 = high)', ylabel='Patients')
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.boxplot(data=df, x='disease_risk', y='age', hue='disease_risk', palette='Set2', legend=False, ax=ax)
ax.set(title='Age distribution by disease risk', xlabel='Disease risk (0 = low, 1 = high)', ylabel='Age (years)')
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(df.corr(numeric_only=True), cmap='coolwarm', center=0, vmin=-1, vmax=1, ax=ax)
ax.set(title='Pearson correlations among features and disease risk', xlabel='Variable', ylabel='Variable')
plt.tight_layout(); plt.show()

## Phases 2–5 — Missingness, outliers, engineering, encoding 
**Order within modeling:** split after safe row checks/EDA, then fit all learned cleaning only on training folds. Missing values are imputed with the **training median**, which is less sensitive to extreme values. We inspect outliers using IQR but keep them: extreme medical measurements may be informative, and arbitrary deletion could alter the target distribution. There is no date, mixed text or new feature required; binary 0/1 fields are already encoded. The pipeline below includes `OneHotEncoder` for robust explicit binary categorical encoding (drop first level), with unknown-category handling. Target is never encoded from feature values.

Random-sample imputation is **not inherently best for linear models**, and KNN imputation is **not guaranteed to outperform mean/median**. Arbitrary/end-of-distribution imputation may signal missingness but can distort scales; missing-not-at-random cannot be inferred from this CSV alone. Compare imputation choices by training-only CV if warranted.

In [ ]:
continuous = ['age','bmi','blood_pressure','cholesterol','blood_sugar','max_heart_rate']
binary = ['sex','exercise_angina','smoking','family_history']
print('Potential IQR outlier counts (diagnostic only):')
for col in continuous:
    q1, q3 = df[col].quantile([.25, .75]); iqr = q3-q1
    count = int(((df[col] < q1-1.5*iqr) | (df[col] > q3+1.5*iqr)).sum())
    print(f'{col}: {count}')
print('No rows removed based only on this diagnostic.')

## Phase 6 — Scale / transform 
`StandardScaler` uses **training-fold** means and standard deviations and is important for distance/margin-sensitive SVM and regularized Logistic Regression. GaussianNB operates on continuous numerical inputs; here it receives the same representation for a controlled comparison. Decision Trees split by ordering and are insensitive to monotonic scaling, so a separate unscaled branch is used. Log, power and binning transforms are unnecessary here and would require validation; no blind transformation is applied.

## Phase 7 — ColumnTransformer and pipelines 
First hold out 20% for final evaluation (`random_state=42`, stratified). The numeric and binary imputation, categorical encoding, and scaling are fitted inside each CV fold, preventing leakage. `add_indicator` and imputation strategies receive a small CV comparison for the scale-sensitive pipelines.

In [ ]:
X = df.drop(columns='disease_risk')
y = df['disease_risk'].astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=.20, stratify=y, random_state=42)
print('Training:', X_train.shape, 'Test:', X_test.shape)
print('Training classes:', y_train.value_counts().sort_index().to_dict())
print('Test classes:', y_test.value_counts().sort_index().to_dict())

def make_preprocessor(scale=True):
    numeric_steps = [('imputer', SimpleImputer(strategy='median', add_indicator=False))]
    if scale:
        numeric_steps.append(('scaler', StandardScaler()))
    return ColumnTransformer([
        ('num', Pipeline(numeric_steps), continuous),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(drop='if_binary', handle_unknown='ignore', sparse_output=False))
        ]), binary)
    ], verbose_feature_names_out=False)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
base_models = {
    'Logistic Regression': LogisticRegression(max_iter=2000, random_state=42),
    'SVM (linear)': SVC(kernel='linear', probability=True, random_state=42),
    'SVM (RBF)': SVC(kernel='rbf', probability=True, random_state=42),
    'Gaussian Naive Bayes': GaussianNB(),
    'Decision Tree': DecisionTreeClassifier(random_state=42)
}
def make_pipeline(name, estimator):
    return Pipeline([('preprocessor', make_preprocessor(scale=name != 'Decision Tree')),
                     ('classifier', estimator)])

## Phase 8 — PCA 
PCA is optional here. Ten meaningful original features and an interpretable tree make dimensionality reduction unnecessary; PCA would make feature interpretation harder. If tried, PCA must be fitted inside CV after numeric imputation/scaling.

## Part C — Default models, then tuning
First fit and evaluate default hyperparameters by **training-only 5-fold CV**. The required SVM is evaluated with both kernels. Then tune at least one parameter per family; Naive Bayes uses `var_smoothing`. Primary model selection uses CV **ROC-AUC**; its threshold-independent ranking is useful while precision and recall remain visible. Test results are inspected once after selection.

In [ ]:
default_rows = []
for name, estimator in base_models.items():
    pipeline = make_pipeline(name, estimator)
    scores = cross_val_score(pipeline, X_train, y_train, scoring='roc_auc', cv=cv, n_jobs=1)
    default_rows.append({'Model': name, 'Default CV ROC-AUC mean': scores.mean(),
                         'Default CV ROC-AUC std': scores.std()})
default_results = pd.DataFrame(default_rows).sort_values('Default CV ROC-AUC mean', ascending=False)
display(default_results.round(3))

In [ ]:
imputation_grid = {
    'preprocessor__num__imputer__strategy': ['mean', 'median'],
    'preprocessor__num__imputer__add_indicator': [False, True]
}
grids = {
    'Logistic Regression': {**imputation_grid, 'classifier__C':[0.1, 1, 10]},
    'SVM (linear)': {**imputation_grid, 'classifier__C':[0.1, 1, 10]},
    'SVM (RBF)': {**imputation_grid, 'classifier__C':[0.1, 1, 10], 'classifier__gamma':['scale', 0.1]},
    'Gaussian Naive Bayes': {'classifier__var_smoothing':[1e-11, 1e-9, 1e-7]},
    'Decision Tree': {'classifier__max_depth':[3, 5, None],
                      'classifier__min_samples_leaf':[1, 5, 10]}
}
searches = {}
cv_rows = []
for name, estimator in base_models.items():
    search = GridSearchCV(make_pipeline(name, estimator), grids[name], scoring='roc_auc',
                          cv=cv, n_jobs=1, refit=True)
    search.fit(X_train, y_train)
    searches[name] = search
    cv_rows.append({'Model':name, 'Best CV ROC-AUC':search.best_score_,
                    'Best parameters':str(search.best_params_)})
cv_results = pd.DataFrame(cv_rows).sort_values('Best CV ROC-AUC', ascending=False)
display(cv_results)

**Parameter-grid note:** categorical imputer strategy/indicator settings are not swept: binary fields contain no missing values, so their setting has no effect. Mean/median and missing-indicator choices are compared on the three incomplete continuous features. A fixed seeded 5-fold split is reused for every candidate.

## Part D — Final held-out evaluation
All metrics refer to high-risk patients (positive label = 1). Precision/recall/F1 use `classification_report`; ROC-AUC uses predicted probabilities. Two SVM rows document both kernels, while the requested four-family table selects whichever SVM kernel performed better **on training CV**.

In [ ]:
test_rows = []
for name, search in searches.items():
    model = search.best_estimator_
    predicted = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]
    print('\n', name, '\n', classification_report(y_test, predicted, target_names=['Low risk', 'High risk'], zero_division=0))
    test_rows.append({'Model':name, 'Accuracy':accuracy_score(y_test,predicted),
                      'Precision':precision_score(y_test,predicted,zero_division=0),
                      'Recall':recall_score(y_test,predicted,zero_division=0),
                      'F1-Score':f1_score(y_test,predicted,zero_division=0),
                      'ROC-AUC':roc_auc_score(y_test,probabilities)})
all_results = pd.DataFrame(test_rows).set_index('Model')
display(all_results.round(3))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, (name, search) in zip(axes.flat, searches.items()):
    ConfusionMatrixDisplay.from_estimator(search.best_estimator_, X_test, y_test,
        display_labels=['Low risk','High risk'], cmap='Blues', colorbar=False, ax=ax)
    ax.set_title(f'{name}: test confusion matrix')
    ax.set_xlabel('Predicted risk'); ax.set_ylabel('Actual risk')
for ax in axes.flat[len(searches):]: ax.axis('off')
fig.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for name, search in searches.items():
    probabilities = search.best_estimator_.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, probabilities)
    ax.plot(fpr, tpr, label=f'{name} (AUC={roc_auc_score(y_test,probabilities):.3f})')
ax.plot([0,1], [0,1], 'k--', label='Chance')
ax.set(title='ROC curves on held-out test patients', xlabel='False positive rate', ylabel='True positive rate', xlim=(0,1), ylim=(0,1))
ax.legend(loc='lower right', fontsize=8); plt.tight_layout(); plt.show()

In [ ]:
best_svm = cv_results.loc[cv_results.Model.str.startswith('SVM')].iloc[0]['Model']
summary = all_results.loc[['Logistic Regression', best_svm, 'Gaussian Naive Bayes', 'Decision Tree']].copy()
summary.index = ['Logistic Regression', f'SVM ({best_svm.split("(")[1]}', 'Naive Bayes', 'Decision Tree']
print('Four-family comparison (SVM kernel chosen by training CV):')
display(summary.round(3))

### Bonus — Decision Tree feature importance
Impurity importance is descriptive and can favor continuous or high-cardinality features. It does not establish clinical causality.

In [ ]:
tree = searches['Decision Tree'].best_estimator_
feature_names = tree.named_steps['preprocessor'].get_feature_names_out()
importance = pd.Series(tree.named_steps['classifier'].feature_importances_, index=feature_names).sort_values()
fig, ax = plt.subplots(figsize=(7, 5))
importance.plot.barh(ax=ax, color='#36769d')
ax.set(title='Tuned Decision Tree feature importance', xlabel='Mean decrease in impurity', ylabel='Encoded feature')
plt.tight_layout(); plt.show()
print('Top tree features:', importance.tail(3).sort_values(ascending=False).round(3).to_dict())

## Part E — Analysis and conclusion (200–300 words)

*The next cell writes a data-specific analysis from the executed results; read the displayed paragraph as the final submission text.*

In [ ]:
cv_order = cv_results.set_index('Model')['Best CV ROC-AUC']
best_name, worst_name = cv_order.idxmax(), cv_order.idxmin()
chosen = 'Logistic Regression' if cv_order['Logistic Regression'] >= cv_order[best_svm] - 0.02 else best_svm
best = all_results.loc[best_name]; worst = all_results.loc[worst_name]
lead_features = ', '.join(importance.tail(3).sort_values(ascending=False).index.tolist())
analysis = f"""Using a stratified 80/20 split, I held out {len(X_test)} patients and selected settings with five-fold cross-validation on the {len(X_train)} training patients. The best model by training cross-validated ROC-AUC was {best_name} ({cv_order[best_name]:.3f}); on the test set it achieved accuracy {best['Accuracy']:.3f}, precision {best['Precision']:.3f}, recall {best['Recall']:.3f}, F1 {best['F1-Score']:.3f}, and ROC-AUC {best['ROC-AUC']:.3f}. These values should be read together: ROC-AUC assesses ranking over thresholds, while recall measures how many high-risk patients were detected at the default threshold. The small dataset mixes continuous measurements with binary risk factors, which can favor a flexible nonlinear boundary or a stable regularized linear boundary depending on the observed relationships; this experiment alone cannot prove the underlying clinical mechanism.

The weakest model by the same training CV criterion was {worst_name} (CV ROC-AUC {cv_order[worst_name]:.3f}; test ROC-AUC {worst['ROC-AUC']:.3f}). Its assumptions or limited data may explain the gap: GaussianNB assumes conditional independence, a shallow tree may miss interactions, and an unrestricted tree may overfit. These are plausible explanations, not established causes. The tree's leading importance values were associated with {lead_features}; impurity importance does not imply causation.

For medical screening, I would prioritize recall to reduce missed high-risk patients, while setting an acceptable false-positive workload with clinicians. At a fixed threshold, improving recall often lowers precision; the reported default-threshold results do not choose an operating point. For a provisional production candidate I would choose {chosen}, considering its training CV ranking alongside speed and ease of explanation. Before any real deployment, I would verify calibration, choose a clinically justified threshold, check subgroup performance, and validate prospectively on external patients. The 80-person test set alone is insufficient to establish clinical reliability."""
print('Word count:', len(analysis.split()))
display(Markdown(analysis))

### Reproducibility notes
Data: supplied `disease_risk_dataset.csv`. `random_state=42`; stratified 80/20 test and seeded 5-fold training CV. All learned imputation/scaling/encoding is inside fitted pipelines. This synthetic-style educational exercise makes no medical claim. Default SVC uses probability calibration internally; its probabilities are computed entirely from training data. To rerun, keep the CSV beside the notebook or in `upload/` and run all cells in order.